# Side-Channel Analysis — CV32E40P RISC-V on CW305
**TU Delft CESE4040 PDP Project**

This notebook documents the complete SCA experiments performed on the CW305 Artix-7 FPGA with a CV32E40P RISC-V soft-core.

## Experiments
1. TVLA — Test Vector Leakage Assessment on three designs
2. CPA — Correlation Power Analysis on AES_100t (reference)
3. CPA — on Zkne Unprotected design
4. CPA — on DOM Protected design

## Hardware Setup
- CW305 Artix-7 XC7A100T FPGA board
- ChipWhisperer-Lite scope
- SMA cable connected to X4 (low-noise measurement point)
- 20-pin ribbon cable between CW305 and ChipWhisperer-Lite
- Clock: 50MHz (10MHz for AES_100t), ADC: extclk_x4

## Cell 1 — Imports and Configuration

In [ ]:
import chipwhisperer as cw
import chipwhisperer.analyzer as cwa
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os
import time
from chipwhisperer.analyzer.attacks.models.aes.key_schedule import key_schedule_rounds

# Paths
AES100T      = r'C:\Users\rishi\AppData\Local\Programs\Python\Python311\Lib\site-packages\chipwhisperer\hardware\firmware\cw305\AES_100t.bit'
BASELINE_BIT = r'C:\Users\rishi\Desktop\pdp\Validation\baseline_cw305_top.bit'
DOM_BIT      = r'C:\Users\rishi\Desktop\pdp\Validation\dom_cw305_top.bit'
ZKNE_BIT     = r'C:\Users\rishi\Desktop\pdp\Validation\zkne_fixed_cw305_top.bit'
RESULTS_DIR  = r'C:\Users\rishi\Desktop\pdp\SCA_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Fixed AES-128 test key (NIST FIPS-197)
KEY = bytearray([0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,
                 0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c])

# Known test vector
PT_TEST  = bytearray([0x32,0x43,0xf6,0xa8,0x88,0x5a,0x30,0x8d,
                      0x31,0x31,0x98,0xa2,0xe0,0x37,0x07,0x34])
CT_EXPECTED = '3925841d02dc09fbdc118597196a0b32'

print('Configuration loaded.')

## Cell 2 — Connect to Hardware

In [ ]:
scope  = cw.scope()
scope.default_setup()
scope.adc.basic_mode  = 'rising_edge'
scope.trigger.triggers = 'tio4'
scope.io.tio1 = 'serial_rx'
scope.io.tio2 = 'serial_tx'
scope.io.hs2  = 'disabled'

target = cw.target(scope, cw.targets.CW305, force=True, fpga_id='100t', platform='cw305')
print('Connected.')

def setup_pll(freq=50e6, gain=25, samples=500):
    target.vccint_set(1.0)
    target.pll.pll_enable_set(True)
    target.pll.pll_outenable_set(False, 0)
    target.pll.pll_outenable_set(True,  1)
    target.pll.pll_outenable_set(False, 2)
    target.pll.pll_outfreq_set(freq, 1)
    target.clkusbautooff = True
    target.clksleeptime  = 1
    scope.clock.adc_src  = 'extclk_x4'
    scope.gain.db        = gain
    scope.adc.samples    = samples
    for i in range(5):
        scope.clock.reset_adc()
        time.sleep(1)
        if scope.clock.adc_locked:
            break
    assert scope.clock.adc_locked, 'ADC lock failed'
    print(f'ADC locked at {scope.clock.adc_freq/1e6:.1f} MHz, gain={gain}dB, samples={samples}')

def flash(bitfile):
    with open(bitfile, 'rb') as f:
        target.fpga_write(0, f.read())
    print(f'Programmed: {target.is_programmed()}')

def sanity_check():
    target.set_key(KEY)
    target.loadInput(PT_TEST)
    scope.arm()
    target.go()
    scope.capture()
    ct = target.readOutput()
    ok = ct.hex() == CT_EXPECTED
    print(f'Sanity check: {"PASS" if ok else "FAIL"} — {ct.hex()}')
    return ok

## Cell 3 — TVLA Helper Functions

In [ ]:
def run_tvla(n_traces=5000):
    FIXED_PT = bytearray([0xda,0x39,0xa3,0xee,0x5e,0x6b,0x4b,0x0d,
                          0x32,0x55,0xbf,0xef,0x95,0x60,0x18,0x90])
    traces_fixed  = []
    traces_random = []

    print(f'Capturing {n_traces} TVLA traces...')
    for i in range(n_traces):
        pt = FIXED_PT if i % 2 == 0 else bytearray(np.random.bytes(16))
        ret = cw.capture_trace(scope, target, pt, KEY)
        if not ret:
            continue
        if i % 2 == 0:
            traces_fixed.append(ret.wave)
        else:
            traces_random.append(ret.wave)
        if i % 1000 == 0:
            print(f'  {i}/{n_traces}')

    tf = np.array(traces_fixed)
    tr = np.array(traces_random)
    t_values = np.array([stats.ttest_ind(tf[:,s], tr[:,s], equal_var=False)[0]
                         for s in range(tf.shape[1])])

    threshold = 4.5
    leaking   = np.sum(np.abs(t_values) > threshold)
    print(f'Max |t-value|: {np.max(np.abs(t_values)):.2f}')
    print(f'Samples exceeding threshold: {leaking}')
    print(f'Leakage detected: {leaking > 0}')
    return t_values

## Cell 4 — TVLA Experiment 1: AES_100t (Unprotected Hardware AES)

In [ ]:
flash(AES100T)
setup_pll(freq=10e6, gain=25, samples=500)
sanity_check()

tvla_aes100t = run_tvla(5000)
np.save(os.path.join(RESULTS_DIR, 'tvla_aes100t.npy'), tvla_aes100t)

plt.figure(figsize=(14,4))
plt.plot(tvla_aes100t[:200])
plt.axhline(y=4.5,  color='r', linestyle='--', label='+4.5')
plt.axhline(y=-4.5, color='r', linestyle='--', label='-4.5')
plt.title(f'TVLA — AES_100t (Unprotected Hardware AES) | max|t|={np.max(np.abs(tvla_aes100t)):.2f}')
plt.xlabel('Sample'); plt.ylabel('t-value'); plt.legend(); plt.grid(True)
plt.savefig(os.path.join(RESULTS_DIR, 'tvla_aes100t.png'), dpi=150)
plt.show()

## Cell 5 — TVLA Experiment 2: Baseline RISCY (Software AES)

In [ ]:
flash(BASELINE_BIT)
setup_pll(freq=50e6, gain=30, samples=2000)
sanity_check()

tvla_baseline = run_tvla(5000)
np.save(os.path.join(RESULTS_DIR, 'tvla_baseline.npy'), tvla_baseline)

plt.figure(figsize=(14,4))
plt.plot(tvla_baseline)
plt.axhline(y=4.5,  color='r', linestyle='--', label='+4.5')
plt.axhline(y=-4.5, color='r', linestyle='--', label='-4.5')
plt.title(f'TVLA — Baseline RISCY (Software AES) | max|t|={np.max(np.abs(tvla_baseline)):.2f}')
plt.xlabel('Sample'); plt.ylabel('t-value'); plt.legend(); plt.grid(True)
plt.savefig(os.path.join(RESULTS_DIR, 'tvla_baseline.png'), dpi=150)
plt.show()

## Cell 6 — TVLA Experiment 3: DOM Protected RISCY

In [ ]:
flash(DOM_BIT)
setup_pll(freq=50e6, gain=30, samples=2000)
sanity_check()

tvla_dom = run_tvla(5000)
np.save(os.path.join(RESULTS_DIR, 'tvla_dom.npy'), tvla_dom)

plt.figure(figsize=(14,4))
plt.plot(tvla_dom)
plt.axhline(y=4.5,  color='r', linestyle='--', label='+4.5')
plt.axhline(y=-4.5, color='r', linestyle='--', label='-4.5')
plt.title(f'TVLA — DOM Protected RISCY | max|t|={np.max(np.abs(tvla_dom)):.2f}')
plt.xlabel('Sample'); plt.ylabel('t-value'); plt.legend(); plt.grid(True)
plt.savefig(os.path.join(RESULTS_DIR, 'tvla_dom.png'), dpi=150)
plt.show()

## Cell 7 — TVLA Comparison Plot

In [ ]:
tvla_aes100t = np.load(os.path.join(RESULTS_DIR, 'tvla_aes100t.npy'))
tvla_baseline = np.load(os.path.join(RESULTS_DIR, 'tvla_baseline.npy'))
tvla_dom      = np.load(os.path.join(RESULTS_DIR, 'tvla_dom.npy'))

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(tvla_aes100t[:200], color='red')
axes[0].axhline(y=4.5,  color='k', linestyle='--')
axes[0].axhline(y=-4.5, color='k', linestyle='--')
axes[0].set_title(f'AES_100t (Unprotected HW AES) | max|t|={np.max(np.abs(tvla_aes100t)):.2f} | LEAKS', fontsize=11)
axes[0].set_ylabel('t-value'); axes[0].set_ylim(-55,55); axes[0].grid(True, alpha=0.3)

axes[1].plot(tvla_baseline, color='orange')
axes[1].axhline(y=4.5,  color='k', linestyle='--')
axes[1].axhline(y=-4.5, color='k', linestyle='--')
axes[1].set_title(f'Baseline RISCY (SW AES) | max|t|={np.max(np.abs(tvla_baseline)):.2f} | MARGINAL', fontsize=11)
axes[1].set_ylabel('t-value'); axes[1].set_ylim(-10,10); axes[1].grid(True, alpha=0.3)

axes[2].plot(tvla_dom, color='green')
axes[2].axhline(y=4.5,  color='k', linestyle='--')
axes[2].axhline(y=-4.5, color='k', linestyle='--')
axes[2].set_title(f'DOM Protected RISCY | max|t|={np.max(np.abs(tvla_dom)):.2f} | PROTECTED', fontsize=11)
axes[2].set_ylabel('t-value'); axes[2].set_xlabel('Sample'); axes[2].set_ylim(-10,10); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tvla_comparison.png'), dpi=150)
plt.show()
print('TVLA comparison plot saved.')

## Cell 8 — CPA Experiment 1: AES_100t (Reference Attack)
This validates the measurement setup. CPA should fully recover the 16-byte key.

In [ ]:
flash(AES100T)
setup_pll(freq=10e6, gain=25, samples=129)
sanity_check()

proj_aes100t = cw.create_project('projects/aes100t', overwrite=True)
ktp = cw.ktp.Basic()

print('Capturing 5000 traces...')
for i in range(5000):
    key, text = ktp.next()
    ret = cw.capture_trace(scope, target, text, key)
    if not ret: continue
    proj_aes100t.traces.append(ret)
    if i % 500 == 0: print(f'  {i}/5000')

attack = cwa.cpa(proj_aes100t, cwa.leakage_models.last_round_state_diff)
results = attack.run()

recv_last = [kguess[0][0] for kguess in results.find_maximums()]
recv_key  = key_schedule_rounds(recv_last, 10, 0)
correct_key = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,
               0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]
matches   = sum(r==c for r,c in zip(recv_key, correct_key))
max_corr  = max(kguess[0][2] for kguess in results.find_maximums())

print(f'Recovered: {" ".join(hex(k) for k in recv_key)}')
print(f'Bytes correct: {matches}/16  Max corr: {max_corr:.4f}')
print(f'Attack successful: {matches==16}')

## Cell 9 — CPA Experiment 2: Zkne Unprotected
**Key insight:** Trigger must be placed AFTER key expansion to capture only the hardware Zkne execution.

In [ ]:
flash(ZKNE_BIT)
setup_pll(freq=50e6, gain=9, samples=500)
sanity_check()

proj_zkne = cw.create_project('projects/zkne_clean', overwrite=True)
collected = 0
attempts  = 0

print('Capturing 5000 clean (non-clipping) traces...')
while collected < 5000:
    pt  = bytearray(np.random.bytes(16))
    ret = cw.capture_trace(scope, target, pt, KEY)
    attempts += 1
    if not ret: continue
    if np.any(np.abs(ret.wave) >= 0.49): continue  # skip clipping
    proj_zkne.traces.append(ret)
    collected += 1
    if collected % 1000 == 0: print(f'  {collected}/5000 (attempts: {attempts})')

traces_zkne = np.array([t.wave for t in proj_zkne.traces])
print(f'Max variance: {np.max(np.var(traces_zkne, axis=0)):.6f} at sample {np.argmax(np.var(traces_zkne, axis=0))}')

plt.figure(figsize=(14,4))
plt.plot(np.var(traces_zkne, axis=0)[:100])
plt.title('Zkne Unprotected — Trace Variance (trigger after key expansion)')
plt.xlabel('Sample'); plt.ylabel('Variance'); plt.grid(True)
plt.savefig(os.path.join(RESULTS_DIR, 'zkne_variance.png'), dpi=150)
plt.show()

attack = cwa.cpa(proj_zkne, cwa.leakage_models.sbox_output)
results = attack.run()

recovered = [kguess[0][0] for kguess in results.find_maximums()]
max_corr  = max(kguess[0][2] for kguess in results.find_maximums())
correct   = sum(r==k for r,k in zip(recovered, list(KEY)))

print(f'Bytes correct: {correct}/16  Max corr: {max_corr:.4f}')
print('PGE per byte:')
for i, kguess in enumerate(results.find_maximums()):
    print(f'  Byte {i:2d}: PGE={kguess[0][1]:4d} corr={kguess[0][2]:.4f} guess=0x{kguess[0][0]:02x} correct=0x{list(KEY)[i]:02x}')

## Cell 10 — CPA Experiment 3: DOM Protected

In [ ]:
flash(DOM_BIT)
setup_pll(freq=50e6, gain=30, samples=500)
sanity_check()

proj_dom = cw.create_project('projects/dom_protected', overwrite=True)
ktp = cw.ktp.Basic()

print('Capturing 5000 traces DOM...')
for i in range(5000):
    key, text = ktp.next()
    ret = cw.capture_trace(scope, target, text, key)
    if not ret: continue
    proj_dom.traces.append(ret)
    if i % 1000 == 0: print(f'  {i}/5000')

attack = cwa.cpa(proj_dom, cwa.leakage_models.last_round_state_diff)
results = attack.run()

recv_last = [kguess[0][0] for kguess in results.find_maximums()]
try:
    recv_key = key_schedule_rounds(recv_last, 10, 0)
except:
    recv_key = recv_last
correct_key = [0x2b,0x7e,0x15,0x16,0x28,0xae,0xd2,0xa6,
               0xab,0xf7,0x15,0x88,0x09,0xcf,0x4f,0x3c]
matches  = sum(r==c for r,c in zip(recv_key, correct_key))
max_corr = max(kguess[0][2] for kguess in results.find_maximums())

print(f'DOM: {matches}/16 bytes correct, max corr={max_corr:.4f}')
print(f'Attack successful: {matches==16}')

## Cell 11 — Final Results Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CPA correlation comparison
designs = ['AES_100t\n(HW AES)', 'Zkne Unprotected\n(Our Design)', 'DOM Protected\n(Our Design)']
correlations = [0.236, 0.070, 0.072]
colors = ['red', 'orange', 'green']
bars = axes[0].bar(designs, correlations, color=colors, width=0.4)
axes[0].axhline(y=0.07, color='gray', linestyle='--', label='Noise floor')
axes[0].set_ylabel('Max CPA Correlation')
axes[0].set_title('CPA Attack Results')
axes[0].set_ylim(0, 0.35)
for bar, val in zip(bars, correlations):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

# TVLA comparison
tvla_aes100t  = np.load(os.path.join(RESULTS_DIR, 'tvla_aes100t.npy'))
tvla_baseline = np.load(os.path.join(RESULTS_DIR, 'tvla_baseline.npy'))
tvla_dom      = np.load(os.path.join(RESULTS_DIR, 'tvla_dom.npy'))

designs_tvla = ['AES_100t', 'Baseline RISCY', 'DOM Protected']
tvals = [np.max(np.abs(tvla_aes100t)), np.max(np.abs(tvla_baseline)), np.max(np.abs(tvla_dom))]
colors_tvla = ['red', 'orange', 'green']
bars2 = axes[1].bar(designs_tvla, tvals, color=colors_tvla, width=0.4)
axes[1].axhline(y=4.5, color='k', linestyle='--', label='Threshold (4.5)')
axes[1].set_ylabel('Max |t-value|')
axes[1].set_title('TVLA Results')
for bar, val in zip(bars2, tvals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'final_summary.png'), dpi=150)
plt.show()

print('='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
print(f'AES_100t:         CPA 16/16 bytes  corr=0.236  TVLA t=49.27  VULNERABLE')
print(f'Baseline RISCY:   CPA PGE->5       corr=0.127  TVLA t=4.69   MARGINAL')
print(f'Zkne Unprotected: CPA PGE->1-3     corr=0.070  TVLA N/A      LEAKS')
print(f'DOM Protected:    CPA 0/16 bytes   corr=0.072  TVLA t=3.98   PROTECTED')
print('='*60)